<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L08-reward-model-audit/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L08-reward-model-audit/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L08-reward-model-audit/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L08-reward-model-audit/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L08 · Train a Bradley-Terry reward model and audit what it really learned

**You will build:** the instruments that turn "this reward model is accurate on held-out pairs" into
a statement you can defend: the Bradley-Terry loss and its gradient, a pairwise accuracy, counterfactual
pairs that isolate one shortcut at a time, best-of-n against the learned reward, and an interval on how
much accuracy a shortcut is worth.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, numpy only · **Prerequisites:** T04-L06-dpo-from-scratch,
T10-L07-calibrated-judges, P04-L07-challenger-from-scratch

The world is **simulated**, so the truth is known: a hidden quality for every answer, annotators with
shortcuts planted on purpose, and a gold reward that ignores them. The lesson measures a *method*; it makes
no claim about how any real reward model or annotator behaves.

By the end you will be able to:

1. Implement the Bradley-Terry loss and its gradient, and check that a fitted model sits where the gradient vanishes.
2. Implement pairwise accuracy with exact ties, and measure held-out accuracy against three different sets of labels.
3. Implement counterfactual pairs that swap or equalise one feature, and measure how much a shortcut is worth to the model.
4. Implement best-of-n selection against the learned reward, and measure the learned and the gold reward of its winners as n grows.
5. Implement a paired bootstrap interval for the accuracy drop, and measure whether a model that was never swayed shows one.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T04-L06`, `T10-L07`, `P04-L07`

**This lesson assumes you can already:**

- read a logistic regression as a sigmoid of a weighted sum, fitted by minimising a log-loss, and a gradient as the slope of that loss
- use numpy 2-D arrays: index rows and columns, take a mean along an axis, and build a boolean mask
- recall what a bootstrap does: resample the data with replacement many times and read an interval off the spread of the results

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290) · Rafailov et al. (arXiv 2305.18290) · *free* — sections 3 and 4 state the Bradley-Terry preference model that exercise 1's loss is the negative log-likelihood of
- [RewardBench 2: Advancing Reward Model Evaluation](https://arxiv.org/abs/2506.01937) · Malik et al. (arXiv 2506.01937) · *free* — the best-of-4 scoring with a 25% random baseline that section 6 reproduces in miniature, and its handling of ties
- [A Long Way to Go: Investigating Length Correlations in RLHF](https://arxiv.org/abs/2310.03716) · Singhal et al. (arXiv 2310.03716) · *free* — the evidence that length is a real shortcut in reward models, the one exercise 3 isolates with a swap
- [Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760) · Gao et al. (arXiv 2210.10760) · *free* — the gold-reward and proxy-reward design, and the best-of-n curves of the gold score, that section 6 and exercise 4 reproduce at small scale
- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · The NumPy project · *free* — 2-D indexing, axis arguments and boolean masks, the few numpy operations every exercise here is written in

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import sys
from dataclasses import dataclass
from typing import Callable

import numpy as np

print("python", sys.version.split()[0])
print("numpy ", np.__version__)

# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 60     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---

# Every check and every demo below runs through `_try`, so pressing Run all before you have written
# a line reaches the progress board at the foot of the notebook instead of stopping at the first stub.
_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, function, where).
_EXERCISES = [("exercise 1", "bt_loss_grad", "section 3"),
              ("exercise 2", "pairwise_accuracy", "section 4"),
              ("exercise 3", "counterfactual_pairs", "section 5"),
              ("exercise 4", "best_of_n", "section 6"),
              ("exercise 5", "paired_drop_interval", "section 7")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The function that raised NotImplementedError: the innermost frame of the traceback."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (`longer_win_rate`)"; several exercises -> "exercises 3 and 4"."""
    if len(labels) > 1 and all(label in _FUNCTION for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} (`{_FUNCTION[label]}`)" if label in _FUNCTION else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken exercise
    never hides the feedback on the others. A demo names the exercises it `needs`: until each has
    passed its check, the demo says which one it is waiting for and skips. Nothing is swallowed:
    every outcome is recorded for the progress board, and a script run exits non-zero there if any
    check came back wrong. A check that runs longer than its limit is stopped and counts as failed,
    so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _FUNCTION:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The setting: a reward model is a score, and a score can be gamed

A **reward model** gives every answer one number. It is trained on **preference pairs**: for the same
prompt, an annotator marked one answer *chosen* and the other *rejected*. The standard model of such
labels is **Bradley-Terry**: the chance that answer *a* is chosen over answer *b* is
`sigmoid(r(a) - r(b))`, where `r` is the reward. Fitting `r` is maximum likelihood, a logistic fit on
the *difference* of the two answers' features. Every RL lesson after this one optimises against such a number.

The question this lesson answers is the one a held-out accuracy cannot: **what did the model learn?**
Public benchmarks (as of 12 November 2026, `claims.yaml`) measure accuracy on large fixed sets: RewardBench 2
scores five of its six domains by whether a model picks the one correct completion out of four (its Ties domain has
a score of its own), so a random model scores 25% on those, and M-RewardBench collects preference instances in 23 languages. Accuracy alone cannot say whether a model won
by reading content or by counting words, so this lesson builds the instruments that can.

The world is **simulated**, so the truth is known:

* every answer has a hidden **quality**; a reward model never sees it, it sees six numbers: four noisy
  **readouts** of the quality, a **length** and a **format** score (length is only weakly related to quality and format not at all, by design);
* the **gold reward** is the hidden quality itself, the same idea as the gold-standard reward model Gao et al. use to study
  over-optimisation (`claims.yaml`). No real project has one; a simulation does;
* annotators prefer the better answer but are also swayed by length and format (planted). A second set of labels,
  from annotators who ignore both, is drawn on the same pairs: a **control** only a simulation can build.

The sway strengths are parameters of a scripted stand-in. The lesson measures a *method*; it makes no claim
about how any real annotator or reward model behaves.

In [ ]:
SEED = 20261112                 # the date this lesson was written; no seed was searched
N_TRAIN, N_TEST = 1000, 2000    # preference pairs to fit on / held-out pairs to audit on
POOL_PROMPTS, POOL_SIZE = 400, 64     # best-of-n: 400 prompts, each with a pool of 64 candidate answers
N_READOUTS = 4                  # noisy content readouts per answer
READOUT_NOISE = 1.0             # the noise on each readout, against a quality with spread 1
LENGTH_PER_QUALITY = 0.3        # better answers are a little longer, honestly (correlation of length with quality)
BETA_QUALITY, BETA_LENGTH, BETA_FORMAT = 2.0, 1.2, 1.0   # how strongly annotators are moved by quality, length, format
RIDGE = 0.01                    # the L2 penalty of the fit
NEWTON_STEPS = 30               # reweighted-least-squares steps of the fit
STEP_BOUND = 1e-9               # the fit counts as converged when its last step moves no weight by more than this
GRAD_BOUND = 1e-8               # ... and sits at a stationary point when no gradient entry exceeds this
BOOT = 400                      # bootstrap replicates per interval
REPLICATES = 20                 # fresh worlds for the repeat experiment in section 7

COLUMNS = ["readout 1", "readout 2", "readout 3", "readout 4", "length", "format"]
READOUT_1, LENGTH, FORMAT = 0, 4, 5          # column numbers


def _raises_value_error(function, *args) -> bool:
    """Given: True when `function(*args)` raises ValueError, which is what the checks below ask of bad inputs."""
    try:
        function(*args)
    except ValueError:
        return True
    return False


def sigmoid(x):
    """The logistic function, stable for large |x| (no overflow)."""
    return 0.5 * (1.0 + np.tanh(0.5 * x))


def make_answers(rng, n):
    """Given: n answers. Returns (hidden quality, features of shape (n, 6)). Every draw is made in a fixed order."""
    quality = rng.normal(size=n)
    readouts = quality[:, None] + READOUT_NOISE * rng.normal(size=(n, N_READOUTS))
    length = LENGTH_PER_QUALITY * quality + np.sqrt(1.0 - LENGTH_PER_QUALITY ** 2) * rng.normal(size=n)
    fmt = (rng.random(n) < 0.5).astype(float)
    return quality, np.column_stack([readouts, length, fmt])


@dataclass
class PairSet:
    """Pairs of answers, a and b, with the labels two sets of annotators gave them."""
    quality_a: np.ndarray
    quality_b: np.ndarray
    feats_a: np.ndarray           # (n, 6)
    feats_b: np.ndarray
    swayed_a_wins: np.ndarray     # bool: the swayed annotators chose answer a
    control_a_wins: np.ndarray    # bool: annotators who ignore length and format chose answer a (same pairs, same coin)

    @property
    def gold_a_wins(self) -> np.ndarray:
        """bool: answer a has the higher hidden quality."""
        return self.quality_a > self.quality_b

    def __len__(self) -> int:
        return len(self.quality_a)


def make_pairs(rng, n) -> PairSet:
    """Given: n pairs. The two annotator groups share each pair's random draw, so they differ only in what sways them."""
    qa, fa = make_answers(rng, n)
    qb, fb = make_answers(rng, n)
    coin = rng.random(n)
    swayed = BETA_QUALITY * (qa - qb) + BETA_LENGTH * (fa[:, LENGTH] - fb[:, LENGTH]) + BETA_FORMAT * (fa[:, FORMAT] - fb[:, FORMAT])
    control = BETA_QUALITY * (qa - qb)
    return PairSet(qa, qb, fa, fb, coin < sigmoid(swayed), coin < sigmoid(control))


def orient(feats_a, feats_b, a_wins):
    """Given: (chosen, rejected) feature arrays, taking a's row where a won and b's row where b won."""
    won = np.asarray(a_wins, dtype=bool)[:, None]
    return np.where(won, feats_a, feats_b), np.where(won, feats_b, feats_a)


def fit_bt(diffs, lam=RIDGE, steps=NEWTON_STEPS):
    """Given: the Bradley-Terry maximum-likelihood fit, with an L2 penalty, by iteratively reweighted least squares.

    `diffs[i]` is the chosen answer's features minus the rejected one's. With margin m = diffs @ w and p = sigmoid(m),
    every step solves a weighted least-squares problem on the working response m + 1/p with weights p (1 - p):
    the same fit as P04-L07's logistic IRLS, specialised to no intercept (a reward is only ever compared with another).
    Returns (weights, size of the last update); the loss is convex, so the fit is the global minimum.
    """
    n, d = diffs.shape
    w = np.zeros(d)
    last = float("inf")
    for _ in range(steps):
        m = diffs @ w
        p = sigmoid(m)
        s = p * (1.0 - p)
        z = m + 1.0 / p
        new = np.linalg.solve((diffs.T * s) @ diffs / n + lam * np.eye(d), (diffs.T * s) @ z / n)
        last = float(np.max(np.abs(new - w)))
        w = new
    return w, last


def gold_pairs(pairs):
    """Given: the pairs ordered by the GOLD reward: (better answer's features, worse answer's features)."""
    return orient(pairs.feats_a, pairs.feats_b, pairs.gold_a_wins)


def pair_credit(score_better, score_worse):
    """Given: the credit each pair earns: 1 if the better answer scores higher, 0.5 on an exact tie, 0 if lower."""
    better, worse = np.asarray(score_better), np.asarray(score_worse)
    return (better > worse).astype(float) + 0.5 * (better == worse)


_world = np.random.default_rng(SEED)
TRAIN = make_pairs(_world, N_TRAIN)
TEST = make_pairs(_world, N_TEST)
POOL_QUALITY, POOL_FEATS = make_answers(_world, POOL_PROMPTS * POOL_SIZE)
POOL_QUALITY = POOL_QUALITY.reshape(POOL_PROMPTS, POOL_SIZE)

W_SWAYED, STEP_SWAYED = fit_bt(np.subtract(*orient(TRAIN.feats_a, TRAIN.feats_b, TRAIN.swayed_a_wins)))
W_CONTROL, STEP_CONTROL = fit_bt(np.subtract(*orient(TRAIN.feats_a, TRAIN.feats_b, TRAIN.control_a_wins)))
MODELS = {"swayed-trained": W_SWAYED, "control-trained": W_CONTROL}

if __name__ == "__main__":
    print(f"{N_TRAIN} training pairs, {N_TEST} held-out pairs, and a pool of {POOL_PROMPTS} prompts x {POOL_SIZE} candidates")
    print(f"planted annotator sway per unit of quality / length / format: {BETA_QUALITY} / {BETA_LENGTH} / {BETA_FORMAT} (the control has 0 on the last two)")
    print(f"the same pairs, labelled twice: the two annotator groups disagree on "
          f"{np.mean(TRAIN.swayed_a_wins != TRAIN.control_a_wins):.1%} of the training pairs")
    print(f"\nlearned weights after {NEWTON_STEPS} reweighted steps (the last step moved no weight by more than {STEP_BOUND:g}: "
          f"{max(STEP_SWAYED, STEP_CONTROL) < STEP_BOUND}):")
    print(f"  {'':16s}" + "".join(f"{c:>11s}" for c in COLUMNS))
    for name, w in MODELS.items():
        print(f"  {name:16s}" + "".join(f"{v:11.2f}" for v in w))

## 2. Reading the weights

The two rows above are two reward models fitted to the *same answers* with the *same code*. They differ only in
who labelled the pairs, so compare their length and format columns: any difference is the annotators' sway,
carried into the model. A real reward model has millions of weights and offers you no such table. The rest of
the lesson builds the audit that works when you cannot read the weights: you can only score answers, and you
can only change the answers.

The audit has four instruments, one per exercise: the **loss** that defines what the fit optimises (exercise 1),
**pairwise accuracy** (exercise 2), **counterfactual pairs** that change one feature and leave the content alone
(exercise 3), and **best-of-n** selection against the learned reward (exercise 4). Exercise 5 puts an interval on the
accuracy a shortcut costs.

## 3. Exercise 1 — `bt_loss_grad`: the loss the fit minimises, and its gradient

`D` has one row per pair: the chosen answer's features minus the rejected answer's. The reward margin of
pair *i* is `m_i = D[i] @ w`, and Bradley-Terry says the chosen answer wins with probability `sigmoid(m_i)`.
The loss is the **mean** over pairs of the negative log-likelihood `-log sigmoid(m_i)`, plus the penalty
`lam / 2` times the sum of the squared weights (every weight is penalised; there is no intercept).

Return the loss and its gradient with respect to `w` as a **tuple** `(loss, grad)`. Two things go wrong silently:
`log(1 + exp(-m))` overflows at `m = -800`, and a gradient that does not match the loss still *looks* fine.
The loss and the gradient must be exact and finite **however large a margin is**: the check below tries margins of plus and
minus 800, 1,500 and a million, and moderate ones (plus 10, minus 10, minus 25 and minus 40), so a margin clipped at any bound, or a loss that
drops its small term below some margin, gives itself away. It tests the match between loss and
gradient too, and asks the strongest question there is: at the weights the fit found, your gradient must vanish.

<details><summary>💡 Hint 1 — what to think about</summary>

`-log sigmoid(m)` is `log(1 + exp(-m))`. numpy has a function that computes `log(exp(a) + exp(b))` without ever forming
the large exponential. What are `a` and `b` here? And what is the derivative of that loss with respect to the margin?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Differentiate the loss of one pair with respect to its margin first, then pass that derivative back through the margin to `w`
(the chain rule: each pair's derivative is carried by its own row of `D`). The loss is a mean over pairs, and the penalty has a
derivative of its own. A `D` with no rows, or a `w` whose length is not the number of columns of `D`, is the caller's mistake: raise `ValueError`.

</details>

In [ ]:
def bt_loss_grad(w, D, lam=0.0):
    """The penalised Bradley-Terry loss of weights `w` on difference rows `D`, and its gradient.

    Args:
        w: weights, shape (d,).
        D: chosen-minus-rejected features, shape (n, d).
        lam: the L2 penalty; the loss includes ``lam / 2 * sum(w ** 2)``.

    The loss is the MEAN of ``-log sigmoid(D @ w)`` over the n pairs plus the penalty. It must be exact and finite however
    large the margins are (plus or minus 800, 1,500, a million). The inputs are not changed. A `D` that is not two-dimensional
    or has no rows, or a `w` whose length is not the number of columns of `D`, raises ``ValueError``.

    Returns:
        A tuple ``(loss, grad)``: the loss as a Python float and the gradient as an array of shape (d,).

    Example:
        >>> loss, grad = bt_loss_grad(np.array([0.0]), np.array([[1.0]]))
        >>> round(loss, 4), grad.tolist()
        (0.6931, [-0.5])
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks: run these as often as you like.
def _check_bt_loss_grad() -> None:
    one = np.array([[1.0]])
    loss, grad = bt_loss_grad(np.array([np.log(3.0)]), one)
    assert abs(loss - np.log(4.0 / 3.0)) < 1e-12 and abs(grad[0] + 0.25) < 1e-12, (
        f"got loss {loss!r}, gradient {grad!r}. A margin of ln 3 means the chosen answer wins with probability 3/4: the loss is "
        "-ln(3/4) and the gradient is -(1 - 3/4) times the row of D.")
    two = np.array([[1.0], [-1.0]])
    loss, grad = bt_loss_grad(np.array([np.log(3.0)]), two)
    assert abs(loss - 0.5 * np.log(16.0 / 3.0)) < 1e-12 and abs(grad[0] - 0.25) < 1e-12, (
        f"got loss {loss!r}, gradient {grad!r}. The loss is the MEAN over pairs, not the sum, and each row of D carries its own sign.")
    loss, grad = bt_loss_grad(np.array([1.0, -2.0]), np.zeros((1, 2)), lam=0.5)
    assert abs(loss - (np.log(2.0) + 1.25)) < 1e-12 and np.allclose(grad, [0.5, -1.0], rtol=0, atol=1e-12), (
        f"got loss {loss!r}, gradient {grad!r}. With lam=0.5 the penalty adds lam/2 * sum(w**2) to the loss and lam * w to the gradient, in both.")
    loss, grad = bt_loss_grad(np.zeros(3), np.array([[1.0, 2.0, 0.0], [3.0, 0.0, -2.0]]))
    assert abs(loss - np.log(2.0)) < 1e-12 and np.allclose(grad, [-1.0, -0.5, 0.5], rtol=0, atol=1e-12), (
        f"got loss {loss!r}, gradient {grad!r} at w = 0 on two pairs. Every margin is 0, so the loss is ln 2 and the gradient is minus half the mean row.")
    for margin_value in (-10.0, -25.0, -40.0, 10.0):
        loss, grad = bt_loss_grad(np.array([margin_value]), one)
        want_loss = max(-margin_value, 0.0) + np.log1p(np.exp(-abs(margin_value)))
        want_grad = -(1.0 / (1.0 + np.exp(margin_value)))
        assert abs(loss - want_loss) < 1e-12 and abs(grad[0] - want_grad) < 1e-12, (
            f"got loss {loss!r}, gradient {grad!r} at a margin of {margin_value:g}; the loss is {want_loss!r} and the gradient {want_grad!r}. "
            "Exact means exact at every margin, not only at huge ones: do not replace log(1 + exp(-m)) by -m for a negative m, "
            "however negative, because the term log(1 + exp(-|m|)) is small but not zero.")
    for scale in (800.0, 1500.0, 1e6):
        for sign in (1.0, -1.0):
            loss, grad = bt_loss_grad(np.array([sign * scale]), two)
            assert np.isfinite(loss) and np.isfinite(grad).all() and abs(loss - scale / 2) < 1e-6 and abs(grad[0] - 0.5 * sign) < 1e-9, (
                f"got loss {loss!r}, gradient {grad!r} at margins of plus and minus {scale:g}. Compute log(1 + exp(-m)) with np.logaddexp(0, -m) "
                f"and the weight sigmoid(-m) without ever forming exp of a large number, and never clip a margin; the correct loss is {scale / 2:g} "
                "and the correct gradient is 0.5 for a positive weight and -0.5 for a negative one.")
    loss, grad = bt_loss_grad(np.array([900.0, -1100.0]), np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 1.0], [2.0, -1.0]]), 0.0)
    assert np.isfinite(loss) and np.isfinite(grad).all() and abs(loss - 775.0) < 1e-9 and np.allclose(grad, [0.25, -0.5], rtol=0, atol=1e-9), (
        f"got loss {loss!r}, gradient {grad!r} for margins 900, -1100, -2000 and 2900. Only the two losing pairs cost anything: (1100 + 2000) / 4 "
        "= 775 for the loss, and the gradient is minus the mean of those two rows, (0.25, -0.5).")
    for seed, n_pairs, d_cols, lam_value in ((1, 9, 3, 0.0), (2, 40, 6, 0.01), (3, 5, 2, 2.0)):
        rng = np.random.default_rng(seed)
        D_ref, w_ref = rng.normal(size=(n_pairs, d_cols)), rng.normal(size=d_cols)
        loss, grad = bt_loss_grad(w_ref, D_ref, lam_value)
        ref_loss, ref_grad = 0.0, np.zeros(d_cols)
        for row in D_ref:
            m = float(row @ w_ref)
            ref_loss += max(-m, 0.0) + np.log1p(np.exp(-abs(m)))
            ref_grad -= (np.exp(-m) / (1.0 + np.exp(-m)) if m >= 0 else 1.0 / (1.0 + np.exp(m))) * row
        ref_loss = ref_loss / n_pairs + 0.5 * lam_value * float(w_ref @ w_ref)
        ref_grad = ref_grad / n_pairs + lam_value * w_ref
        assert abs(loss - ref_loss) < 1e-10 and np.allclose(grad, ref_grad, rtol=0, atol=1e-10), (
            f"seed {seed}: got ({loss!r}, {grad!r}), the definition gives ({ref_loss!r}, {ref_grad!r}). The loss is the mean over pairs of "
            "-log sigmoid(D @ w) plus lam/2 * sum(w**2), and the gradient is the gradient of exactly that.")
    rng = np.random.default_rng(3)
    D, w = rng.normal(size=(7, 3)), rng.normal(size=3)
    loss, grad = bt_loss_grad(w, D, lam=0.3)
    numeric = np.array([(bt_loss_grad(w + h, D, 0.3)[0] - bt_loss_grad(w - h, D, 0.3)[0]) / 2e-6 for h in 1e-6 * np.eye(3)])
    assert np.allclose(grad, numeric, rtol=0, atol=1e-6), (
        f"the gradient {grad} does not match the slope of your own loss {numeric}. Differentiate the loss you return, penalty included.")
    got = bt_loss_grad(w, D, lam=0.3)
    assert type(got) is tuple and len(got) == 2, f"return a tuple (loss, grad); got a {type(got).__name__}"
    assert type(loss) is float and isinstance(grad, np.ndarray) and grad.shape == (3,), (
        f"return (Python float, array of shape (3,)); got ({type(loss).__name__}, {type(grad).__name__})")
    before_w, before_D = w.copy(), D.copy()
    bt_loss_grad(w, D, 0.3)
    assert (w == before_w).all() and (D == before_D).all(), "bt_loss_grad changed one of its inputs; work on copies"
    for bad in ((np.zeros(2), np.zeros((0, 2))), (np.zeros(3), np.zeros((4, 2))), (np.zeros(2), np.zeros(2)), (np.zeros(2), np.zeros((2, 2, 2)))):
        assert _raises_value_error(bt_loss_grad, *bad), (
            "a D with no rows or one dimension, or a w of the wrong length, must raise ValueError")
    for name, fitted in MODELS.items():
        d_fit = np.subtract(*orient(TRAIN.feats_a, TRAIN.feats_b, TRAIN.swayed_a_wins if name.startswith("swayed") else TRAIN.control_a_wins))
        norm = float(np.max(np.abs(bt_loss_grad(fitted, d_fit, RIDGE)[1])))
        assert norm < GRAD_BOUND, (
            f"at the {name} fit your gradient has a largest entry of {norm:.2e}; the fit sits where the gradient of THIS loss "
            f"(mean over pairs, penalty RIDGE = {RIDGE}) vanishes, so a loss or a gradient that differs from it shows up here.")
    print("exercise 1 looks right")


_try("exercise 1", _check_bt_loss_grad)

In [ ]:
def _show_stationarity() -> None:
    for name, key in (("swayed-trained", TRAIN.swayed_a_wins), ("control-trained", TRAIN.control_a_wins)):
        d_fit = np.subtract(*orient(TRAIN.feats_a, TRAIN.feats_b, key))
        loss0, _ = bt_loss_grad(np.zeros(d_fit.shape[1]), d_fit, RIDGE)
        loss, grad = bt_loss_grad(MODELS[name], d_fit, RIDGE)
        _, grad_unpenalised = bt_loss_grad(MODELS[name], d_fit, 0.0)
        print(f"{name:16s} loss {loss0:.4f} at w = 0 (ln 2 = {np.log(2.0):.4f}) -> {loss:.4f} at the fit; "
              f"no gradient entry above {GRAD_BOUND:g} with the penalty: {np.max(np.abs(grad)) < GRAD_BOUND}; "
              f"largest entry {np.max(np.abs(grad_unpenalised)):.1e} if you forget it")


_try("stationarity", _show_stationarity, needs=("exercise 1",))

## 4. Exercise 2 — `pairwise_accuracy`: the number everyone reports

Given a score for the better answer and a score for the worse answer of each pair, **pairwise accuracy** is the share
of pairs where the better answer scores higher. A pair where the two scores are **exactly equal** earns half credit,
the convention RewardBench 2 uses for a two-way tie among a generative model's ratings (`claims.yaml`); equal means `==`, not close to. A model that scores everything the
same earns 0.5 by this rule, which is what a coin flip would earn. Scores are any real numbers, negative ones included: compare them as
they are, not by their sizes.

"Better" depends on whose labels you use. The next cell measures the same model against three sets of labels on held-out
pairs: the swayed annotators', the control annotators', and the gold reward's.

<details><summary>💡 Hint 1 — what to think about</summary>

Three outcomes per pair: higher, equal, lower. Which of them earns 1, which 0.5, which 0? What should happen to an empty set of pairs?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Turn every pair into its credit (comparing the two arrays gives all the outcomes at once), then average the credits over the pairs.
Every pair counts once, whatever scores it shares with other pairs. Return a Python float. Two arrays of different shapes, arrays
that are not one-dimensional, or no pairs at all are the caller's mistake: raise `ValueError`.

</details>

In [ ]:
def pairwise_accuracy(score_better, score_worse):
    """The share of pairs where the better answer scores higher, with exact ties worth half.

    Args:
        score_better: scores of the better answer of each pair, shape (n,).
        score_worse: scores of the worse answer, same shape.

    A tie is exact equality (``==``). Scores may be negative and are compared as they are. Arrays of different shapes, arrays
    that are not one-dimensional, or no pairs raise ``ValueError``.

    Returns:
        The accuracy, between 0 and 1, as a Python float.

    Example:
        >>> pairwise_accuracy(np.array([2.0, 1.0]), np.array([1.0, 1.0]))
        0.75
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pairwise_accuracy() -> None:
    got = pairwise_accuracy(np.array([2.0, 1.0, 3.0, 5.0]), np.array([1.0, 1.0, 4.0, 0.0]))
    assert abs(got - 0.625) < 1e-12, (
        f"got {got!r}; the four pairs are a win, an exact tie, a loss and a win: (1 + 0.5 + 0 + 1) / 4 = 0.625. "
        "A tie is worth half, not a win and not a loss.")
    assert abs(pairwise_accuracy(np.array([1.0, 2.0, 3.0]), np.array([0.0, 0.0, 0.0])) - 1.0) < 1e-12, (
        "the better answer scores higher on every pair: accuracy 1")
    assert abs(pairwise_accuracy(np.array([0.0, 0.0, 0.0]), np.array([1.0, 2.0, 3.0])) - 0.0) < 1e-12, (
        "the better answer scores lower on every pair: accuracy 0, however large the margins")
    assert abs(pairwise_accuracy(np.array([-3.0, -1.0]), np.array([-4.0, -2.0])) - 1.0) < 1e-12, (
        "negative scores compare as numbers: -3 is higher than -4, so both pairs are wins; do not compare their sizes (abs)")
    assert abs(pairwise_accuracy(np.zeros(5), np.zeros(5)) - 0.5) < 1e-12, (
        "a model that scores every answer the same has only ties, and ties are worth half: 0.5")
    assert abs(pairwise_accuracy(np.array([0.1 + 0.2]), np.array([0.3])) - 1.0) < 1e-12, (
        "0.1 + 0.2 is larger than 0.3 in floating point by 5.6e-17. Ties are exact equality: compare with >, ==, not isclose.")
    one_ulp = float(np.nextafter(1.0, 2.0))
    assert abs(pairwise_accuracy(np.array([one_ulp]), np.array([1.0])) - 1.0) < 1e-12 and abs(pairwise_accuracy(np.array([1.0]), np.array([one_ulp]))) < 1e-12, (
        "two scores one floating-point step apart are not equal: the larger one wins, there is no tolerance")
    assert abs(pairwise_accuracy(np.array([1, 2, 3]), np.array([3, 2, 1])) - 0.5) < 1e-12, "integer scores work the same way"
    for seed, n_pairs in ((6, 200), (7, 17)):
        rng = np.random.default_rng(seed)
        b_scores, w_scores = rng.integers(0, 3, n_pairs), rng.integers(0, 3, n_pairs)
        want = sum(1.0 if x > y else 0.5 if x == y else 0.0 for x, y in zip(b_scores, w_scores)) / n_pairs
        got = pairwise_accuracy(b_scores, w_scores)
        assert abs(got - want) < 1e-12, (
            f"got {got!r} on {n_pairs} pairs with only the scores 0, 1 and 2, so many pairs share a score or are identical; a plain loop gives {want!r}. "
            "Every pair counts once, even when another pair has the same scores: do not collapse pairs through a dict or a set, and do not count unique values.")
    assert type(pairwise_accuracy(np.array([1.0]), np.array([0.0]))) is float, "return a Python float"
    for bad in ((np.zeros(2), np.zeros(3)), (np.zeros((2, 2)), np.zeros((2, 2))), (np.array(0.5), np.array(0.5)), (np.zeros(0), np.zeros(0))):
        assert _raises_value_error(pairwise_accuracy, *bad), (
            "different shapes, arrays that are not one-dimensional, and no pairs must raise ValueError")
    print("exercise 2 looks right")


_try("exercise 2", _check_pairwise_accuracy)

In [ ]:
def _show_accuracy() -> None:
    gold_ceiling = {"swayed": np.mean(TEST.swayed_a_wins == TEST.gold_a_wins), "control": np.mean(TEST.control_a_wins == TEST.gold_a_wins)}
    print(f"on the {N_TEST} held-out pairs the swayed annotators agree with the gold reward on {gold_ceiling['swayed']:.1%} of them, "
          f"the control annotators on {gold_ceiling['control']:.1%}\n")
    print(f"  {'pairwise accuracy':18s}{'vs swayed labels':>18s}{'vs control labels':>19s}{'vs gold':>10s}")
    for name, w in MODELS.items():
        row = []
        for a_wins in (TEST.swayed_a_wins, TEST.control_a_wins, TEST.gold_a_wins):
            chosen, rejected = orient(TEST.feats_a, TEST.feats_b, a_wins)
            row.append(pairwise_accuracy(chosen @ w, rejected @ w))
        print(f"  {name:18s}{row[0]:18.3f}{row[1]:19.3f}{row[2]:10.3f}")
    s_vs_swayed = pairwise_accuracy(*(x @ W_SWAYED for x in orient(TEST.feats_a, TEST.feats_b, TEST.swayed_a_wins)))
    s_vs_gold = pairwise_accuracy(*(x @ W_SWAYED for x in orient(TEST.feats_a, TEST.feats_b, TEST.gold_a_wins)))
    c_vs_gold = pairwise_accuracy(*(x @ W_CONTROL for x in orient(TEST.feats_a, TEST.feats_b, TEST.gold_a_wins)))
    print(f"\nthe swayed model scores {s_vs_swayed - s_vs_gold:+.3f} higher against the labels it was trained to imitate than against the gold reward; "
          f"against the gold reward the control model is {c_vs_gold - s_vs_gold:+.3f} ahead")
    print("a held-out accuracy depends on whose labels grade it, so a single accuracy cannot say whether a model reads content.")


_try("accuracy", _show_accuracy, needs=("exercise 2",))

## 5. Exercise 3 — `counterfactual_pairs`: change one feature, keep the content

To find out whether the model uses length, change the length and leave everything else alone. In a pair
ordered by the gold reward (`gold_pairs` gives the better answer's features and the worse answer's), there are two clean ways:

* **swap** the feature: the better answer gets the worse answer's value and vice versa. If the model had been leaning
  on the feature, and the feature differs systematically between better and worse answers, the lean now points the wrong way and
  accuracy falls. A feature that is unrelated to quality gives the model nothing to lose when it is swapped.
* **equalise** the feature: both answers get the **mean of the two** values. The feature can no longer separate them.

`better` and `worse` are arrays of shape `(n, d)`, row *i* of each belonging to pair *i*. Return the two changed arrays
as a **tuple** `(new_better, new_worse)` of new **float** arrays; every other column and the arrays you were given stay untouched.
`mode` must be exactly `"swap"` or `"equalise"`: any other string, capitals and spelling variants included, raises `ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

What happens to the original arrays if you assign into a slice of a view of them? What dtype does a mean of two integers need?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Copy both arrays as floats first. For a swap, exchange the one column's values between the two copies (take care that the second
assignment does not read what the first already overwrote). For an equalise, both copies get the mean of the pair's two values in that
column. A mode other than exactly "swap" or "equalise" (capital letters do not count), a `column` below 0 or not below the number of
columns, or two arrays that are not two-dimensional and of the same shape, raise `ValueError`.

</details>

In [ ]:
def counterfactual_pairs(better, worse, column, mode):
    """Change one feature of every pair, by swapping it or equalising it.

    Args:
        better, worse: feature arrays of shape (n, d); row i of each is one pair.
        column: the feature to change, with 0 <= column < d.
        mode: exactly ``"swap"`` (the two answers exchange that feature's values) or ``"equalise"`` (both take the mean of the two).

    Returns:
        A tuple ``(new_better, new_worse)`` of new float arrays of shape (n, d). The inputs are not changed and the new arrays
        share no memory with them. A `mode` that is neither of the two (``"Swap"`` is not ``"swap"``), a `column` outside 0..d-1, or arrays that are not
        two-dimensional with the same shape raise ``ValueError``, in either mode.

    Example:
        >>> nb, nw = counterfactual_pairs(np.array([[1, 2]]), np.array([[4, 8]]), 1, "swap")
        >>> nb.tolist(), nw.tolist()
        ([[1.0, 8.0]], [[4.0, 2.0]])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_counterfactual_pairs() -> None:
    better = np.array([[1, 2, 3], [4, 5, 6]])
    worse = np.array([[7, 8, 9], [1, 2, 3]])
    kept_better, kept_worse = better.copy(), worse.copy()
    result = counterfactual_pairs(better, worse, 1, "swap")
    assert type(result) is tuple and len(result) == 2, f"return a tuple (new_better, new_worse); got a {type(result).__name__}"
    nb, nw = result
    assert nb.tolist() == [[1, 8, 3], [4, 2, 6]] and nw.tolist() == [[7, 2, 9], [1, 5, 3]], (
        f"swap of column 1 should give {[[1, 8, 3], [4, 2, 6]]} and {[[7, 2, 9], [1, 5, 3]]}; got {nb.tolist()} and {nw.tolist()}. "
        "Only that column changes, and the two answers exchange values: both must end up with the OTHER one's original value.")
    nb, nw = counterfactual_pairs(better, worse, 2, "swap")
    assert nb.tolist() == [[1, 2, 9], [4, 5, 3]] and nw.tolist() == [[7, 8, 3], [1, 2, 6]], "the last column is a valid column: swap it"
    nb, nw = counterfactual_pairs(better, worse, 0, "equalise")
    assert nb.tolist() == [[4, 2, 3], [2.5, 5, 6]] and nw.tolist() == [[4, 8, 9], [2.5, 2, 3]], (
        f"equalise of column 0 should give {[[4, 2, 3], [2.5, 5, 6]]} and {[[4, 8, 9], [2.5, 2, 3]]}; got {nb.tolist()} and {nw.tolist()}. "
        "Both answers take the mean of the pair's two values, which for integers needs floats (the second pair's mean is 2.5).")
    nb, nw = counterfactual_pairs(np.array([[0.5, 1.0]]), np.array([[0.25, 3.0]]), 1, "equalise")
    assert nb.tolist() == [[0.5, 2.0]] and nw.tolist() == [[0.25, 2.0]], "float input, equalise column 1: both answers take 2.0, the mean of 1.0 and 3.0"
    rng = np.random.default_rng(8)
    many_better, many_worse = rng.normal(size=(30, 5)), rng.normal(size=(30, 5))
    for column in range(5):
        for mode in ("swap", "equalise"):
            nb, nw = counterfactual_pairs(many_better, many_worse, column, mode)
            want_b, want_w = many_better.copy(), many_worse.copy()
            for i in range(30):
                x, y = many_better[i, column], many_worse[i, column]
                want_b[i, column], want_w[i, column] = (y, x) if mode == "swap" else ((x + y) / 2.0, (x + y) / 2.0)
            assert np.allclose(nb, want_b, rtol=0, atol=1e-12) and np.allclose(nw, want_w, rtol=0, atol=1e-12), (
                f"column {column}, {mode}: only that column changes, row by row; a swap exchanges the two values, an equalise gives both the pair's mean")
    for dtype in (int, float):
        ints_better = np.array([[1, 2, 3], [4, 5, 6], [0, 1, 2]], dtype=dtype)
        ints_worse = np.array([[7, 8, 9], [1, 2, 3], [3, 3, 3]], dtype=dtype)
        kept_b, kept_w = ints_better.copy(), ints_worse.copy()
        for column, mode in ((0, "swap"), (1, "equalise"), (2, "swap"), (2, "equalise")):
            nb, nw = counterfactual_pairs(ints_better, ints_worse, column, mode)
            faults = []
            if not ((ints_better == kept_b).all() and (ints_worse == kept_w).all()):
                faults.append("the arrays you were given were changed; copy them first (np.asarray does not copy a float array, np.array does)")
            if nb.dtype != float or nw.dtype != float:
                faults.append(f"return float arrays; got {nb.dtype} and {nw.dtype}")
            if np.shares_memory(nb, ints_better) or np.shares_memory(nw, ints_worse) or np.shares_memory(nb, nw):
                faults.append("the returned arrays must be new arrays, not views of the inputs or of each other")
            assert not faults, f"{np.dtype(dtype).name} input, column {column}, {mode}: " + "; ".join(faults)
    assert (better == kept_better).all() and (worse == kept_worse).all(), "the arrays you were given were changed; copy them first"
    bad_calls = [(better, worse, 1, "swop"), (better, worse, 1, "equalize"), (better, worse, 1, "Swap"), (better, worse, 1, "EQUALISE"), (better, worse, 1, " swap")]
    for mode in ("swap", "equalise"):
        bad_calls += [(better, worse, 3, mode), (better, worse, -1, mode), (better, worse[:1], 1, mode), (better[0], worse[0], 0, mode),
                      (np.zeros((2, 2, 2)), np.ones((2, 2, 2)), 1, mode)]
    for bad in bad_calls:
        assert _raises_value_error(counterfactual_pairs, *bad), (
            f"counterfactual_pairs{bad[2:]} must raise ValueError: a mode that is not exactly 'swap' or 'equalise' (no capitals, no spaces), "
            "a column outside 0..d-1 (negative included) or arrays that are not two-dimensional with one shape, in either mode")
    print("exercise 3 looks right")


_try("exercise 3", _check_counterfactual_pairs)

In [ ]:
CASES = [("readout 1", READOUT_1, "swap"), ("readout 1", READOUT_1, "equalise"),
         ("length", LENGTH, "swap"), ("length", LENGTH, "equalise"),
         ("format", FORMAT, "swap"), ("format", FORMAT, "equalise")]


def _show_counterfactuals() -> None:
    better, worse = gold_pairs(TEST)
    print(f"pairwise accuracy against the gold reward on {N_TEST} held-out pairs, and after changing one feature:\n")
    print(f"  {'':16s}{'as is':>8s}" + "".join(f"{label + ' ' + mode:>20s}" for label, _, mode in CASES))
    for name, w in MODELS.items():
        base = pairwise_accuracy(better @ w, worse @ w)
        cells = []
        for _, column, mode in CASES:
            nb, nw = counterfactual_pairs(better, worse, column, mode)
            cells.append(pairwise_accuracy(nb @ w, nw @ w) - base)
        print(f"  {name:16s}{base:8.3f}" + "".join(f"{c:+20.3f}" for c in cells))
    print("(entries after 'as is' are the CHANGE in accuracy; the readout 1 columns are a reference: that feature really carries content)")


_try("counterfactuals", _show_counterfactuals, needs=("exercise 2", "exercise 3"))


def _show_lean() -> None:
    better, worse = gold_pairs(TEST)
    print("why a swap costs what it costs: the weight, times how much the feature differs between better and worse answers (the gap):\n")
    print(f"  {'':16s}{'feature':>12s}{'weight':>9s}{'gap':>9s}{'weight x gap':>14s}{'swap change':>13s}{'equalise change':>17s}")
    for name, w in MODELS.items():
        base = pairwise_accuracy(better @ w, worse @ w)
        for label, column in (("readout 1", READOUT_1), ("length", LENGTH), ("format", FORMAT)):
            gap = float(np.mean(better[:, column] - worse[:, column]))
            changes = []
            for mode in ("swap", "equalise"):
                nb, nw = counterfactual_pairs(better, worse, column, mode)
                changes.append(pairwise_accuracy(nb @ w, nw @ w) - base)
            print(f"  {name:16s}{label:>12s}{w[column]:+9.2f}{gap:+9.2f}{w[column] * gap:+14.2f}{changes[0]:+13.3f}{changes[1]:+17.3f}")
    print("\nthe gap is the mean of (better answer's value minus worse answer's value) of the feature, over the held-out gold-ordered pairs.")
    print("a swap lowers accuracy where weight x gap is large and positive; a feature whose gap is near zero costs nothing to swap however large its weight,")
    print("and a model's lean on such a feature, if it has one, shows in the equalise column instead: accuracy gained once the feature can no longer sway the score.")


_try("lean", _show_lean, needs=("exercise 2", "exercise 3"))

## 6. Exercise 4 — `best_of_n`: optimise against the reward and watch what you get

Held-out accuracy asks the model to rank pairs. What an RL lesson does with it is different: it asks for the
answer the reward likes best. **Best-of-n** is the simplest version: from a pool of candidate answers to one prompt,
keep the one with the highest learned reward. The gold reward of what was kept tells you what the reward model was worth.

`learned` and `measured` are arrays of shape `(prompts, pool)`: for every candidate, its learned reward and any quantity
you want to track (its gold reward, its length, whether it is the best of the four). Use only the **first `n` candidates**
of each prompt, pick the one with the highest learned reward (on a tie, the one with the lowest index), and return, as a **tuple**,
the mean of the learned reward and the mean of the measured quantity over those winners.

<details><summary>💡 Hint 1 — what to think about</summary>

Which columns are allowed to compete when `n` is smaller than the pool? Which array decides the winner, and which is only read at the winner?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Decide each prompt's winner from the eligible learned rewards only, and note where in the row it sits. Read the measured quantity at that same place,
not at its own best. Think about which candidate should win when two eligible rewards are equal. Average over prompts. An `n` below 1 or above the pool size, arrays of different shapes,
arrays that are not two-dimensional, or no prompts at all raise `ValueError`.

</details>

In [ ]:
def best_of_n(learned, measured, n):
    """Mean learned reward and mean measured quantity of the winners of best-of-n.

    Args:
        learned: shape (prompts, pool), the learned reward of every candidate.
        measured: shape (prompts, pool), any quantity to read at the winners.
        n: how many candidates compete; only columns 0..n-1 of each row are eligible.

    The winner of a row is its eligible candidate with the highest learned reward, the lowest index on a tie. An `n`
    outside 1..pool, arrays that differ in shape (in rows or in columns), arrays that are not two-dimensional, or no prompts
    raise ``ValueError``.

    Returns:
        A tuple ``(mean learned reward of the winners, mean measured quantity of the winners)`` of two Python floats.

    Example:
        >>> best_of_n(np.array([[1.0, 5.0]]), np.array([[10.0, 20.0]]), 2)
        (5.0, 20.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_of_n() -> None:
    learned = np.array([[1.0, 5.0, 3.0], [2.0, 2.0, 9.0]])
    measured = np.array([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]])
    for n, want in ((1, (1.5, 25.0)), (2, (3.5, 30.0)), (3, (7.0, 40.0))):
        got = best_of_n(learned, measured, n)
        assert type(got) is tuple and len(got) == 2, f"return a tuple (mean learned reward, mean measured quantity); got a {type(got).__name__}"
        assert all(abs(g - x) < 1e-12 for g, x in zip(got, want)), (
            f"n={n}: got {got}, expected {want}. Only the first n candidates compete; the winner is the highest LEARNED reward "
            "(the lowest index on a tie), and the measured quantity is read at the winner, not maximised on its own.")
    got = best_of_n(np.array([[0.0, 9.0, 1.0]]), np.array([[7.0, -3.0, 100.0]]), 3)
    assert got == (9.0, -3.0), f"got {got}: the measured quantity is read where the learned reward is highest, even if it is low there"
    three = best_of_n(np.array([[1.0, 2.0], [3.0, 1.0], [5.0, 0.0]]), np.array([[0.0, 10.0], [20.0, 0.0], [100.0, 0.0]]), 2)
    assert abs(three[0] - 10.0 / 3.0) < 1e-12 and abs(three[1] - 130.0 / 3.0) < 1e-12, (
        f"got {three}, expected (3.33.., 43.33..): the winners' learned rewards are 2, 3, 5 and their measured values 10, 20, 100; return the MEAN of each")
    got = best_of_n(np.array([[2.0, 2.0, 2.0], [1.0, 3.0, 3.0]]), np.array([[4.0, 8.0, 12.0], [5.0, 6.0, 7.0]]), 3)
    assert all(abs(g - x) < 1e-12 for g, x in zip(got, (2.5, 5.0))), (
        f"got {got}, expected (2.5, 5.0): on a tie in the learned reward the winner is the candidate with the lowest index (the measured values 4 and 6)")
    rng = np.random.default_rng(9)
    for many_learned, many_measured in ((rng.normal(size=(40, 8)), rng.normal(size=(40, 8))),
                                        (rng.integers(0, 3, (40, 8)).astype(float), rng.normal(size=(40, 8)))):
        for n in range(1, 9):
            want = [0.0, 0.0]
            for row_l, row_m in zip(many_learned, many_measured):
                best = 0
                for j in range(1, n):
                    if row_l[j] > row_l[best]:
                        best = j
                want[0] += row_l[best] / 40
                want[1] += row_m[best] / 40
            got = best_of_n(many_learned, many_measured, n)
            assert all(abs(g - x) < 1e-12 for g, x in zip(got, want)), (
                f"n={n}: got {got}, a plain loop over the first n candidates gives {tuple(want)}: candidates beyond n are not eligible, "
                "and on a tie the lowest index wins")
    one_row_l, one_row_m = np.array([[1.0, 2.0, 3.0]]), np.array([[3.0, 2.0, 1.0]])
    assert all(type(v) is float for v in best_of_n(one_row_l, one_row_m, 2)), "return Python floats (wrap each with float())"
    for bad in ((one_row_l, one_row_m, 0), (one_row_l, one_row_m, 4), (learned, measured[:1], 2), (one_row_l, np.zeros((1, 4)), 2),
                (one_row_l[0], one_row_m[0], 1), (np.zeros((1, 2, 2)), np.zeros((1, 2, 2)), 1), (np.zeros((0, 3)), np.zeros((0, 3)), 1)):
        assert _raises_value_error(best_of_n, *bad), (
            "n below 1 or above the pool, arrays that differ in rows or in columns, a one-dimensional array and no prompts "
                             "must raise ValueError")
    print("exercise 4 looks right")


_try("exercise 4", _check_best_of_n)

In [ ]:
NS = (1, 2, 4, 8, 16, 32, 64)


def _show_best_of_n() -> None:
    gold = POOL_QUALITY
    length = POOL_FEATS[:, LENGTH].reshape(POOL_PROMPTS, POOL_SIZE)
    print(f"best-of-n over {POOL_PROMPTS} prompts, pools of {POOL_SIZE}; 'gain' is in standard deviations of the pool's own scores since n = 1\n")
    summary, gold_curves = {}, {}
    for name, w in MODELS.items():
        learned = (POOL_FEATS @ w).reshape(POOL_PROMPTS, POOL_SIZE)
        gold_curves[name] = []
        print(f"  {name}")
        print(f"    {'n':>3s}{'learned':>9s}{'gold':>8s}{'length':>8s}{'learned gain':>14s}{'gold gain':>11s}")
        base = best_of_n(learned, gold, 1)
        for n in NS:
            mean_learned, mean_gold = best_of_n(learned, gold, n)
            mean_length = best_of_n(learned, length, n)[1]
            gold_curves[name].append(mean_gold)
            print(f"    {n:3d}{mean_learned:9.2f}{mean_gold:8.2f}{mean_length:8.2f}"
                  f"{(mean_learned - base[0]) / learned.std():14.2f}{(mean_gold - base[1]) / gold.std():11.2f}")
        summary[name] = (mean_learned, mean_gold, mean_length, (mean_learned - base[0]) / learned.std(), (mean_gold - base[1]) / gold.std())
    s, c = summary["swayed-trained"], summary["control-trained"]
    print(f"\nat n = {NS[-1]} the swayed model's winners have a gold reward {s[1] - c[1]:+.2f} against the control's, and are {s[2] - c[2]:+.2f} longer")
    print(f"the learned reward's gain minus the gold reward's gain at n = {NS[-1]}: {s[3] - s[4]:+.2f} sd for the swayed model, {c[3] - c[4]:+.2f} sd for the control")
    print(f"the share of the learned gain that shows up as gold gain at n = {NS[-1]}: {s[4] / s[3]:.0%} for the swayed model, {c[4] / c[3]:.0%} for the control")
    for name, curve in gold_curves.items():
        print(f"the gold reward of the {name} winners never fell as n doubled from {NS[0]} to {NS[-1]}: {all(b >= a for a, b in zip(curve, curve[1:]))}")


_try("best-of-n", _show_best_of_n, needs=("exercise 4",))

### Reading the curves

Two things in the table are expected before you look. A learned reward that agrees with the gold reward at less than 1 gains
more on itself than on the gold reward when you select on it, so a learned gain above the gold gain is regression to the mean, not a finding.
What can differ, and what the lines above measure, is **how much of the learned gain reaches the gold reward** and **whether the gold
reward ever falls** as n grows. Gao et al. report that optimising a proxy too hard can lower the gold reward (`claims.yaml`). Here the proxy is a
linear function of the features and none of them misleads only in the tails, so the winners' gold reward has no particular reason to turn down;
the printed check says whether it did on this seed. That is a statement about this small world, not about the paper's: a world where
the proxy is wrong only for rare extreme answers is where a decline could appear, and this one is not built that way.

### The same instrument, as a RewardBench-2-style score

RewardBench 2 asks a model to pick the one correct completion out of four. Here that is best-of-4 with the measured quantity
"is this candidate the best of the four by the gold reward", so the mean over winners is the accuracy. A random reward gets it right
one time in four; the next cell measures that baseline with random scores instead of quoting it.

In [ ]:
def _show_best_of_4() -> None:
    pool4 = POOL_QUALITY[:, :4]
    is_best = (pool4 == pool4.max(axis=1, keepdims=True)).astype(float)
    rng = np.random.default_rng(SEED + 7)
    random_runs = [best_of_n(rng.random(is_best.shape), is_best, 4)[1] for _ in range(50)]
    print(f"best-of-4 accuracy over {POOL_PROMPTS} prompts (one correct answer in four):")
    for name, w in MODELS.items():
        learned = (POOL_FEATS @ w).reshape(POOL_PROMPTS, POOL_SIZE)[:, :4]
        print(f"  {name:16s}{best_of_n(learned, is_best, 4)[1]:.3f}")
    print(f"  random scores   {np.mean(random_runs):.3f} on average over {len(random_runs)} random draws (exact value for one correct in four: {1 / 4:.3f})")


_try("best-of-4", _show_best_of_4, needs=("exercise 4",))

## 7. Exercise 5 — `paired_drop_interval`: is the drop real?

A counterfactual accuracy is a number from a finite set of held-out pairs; the drop between two of them has noise. Section 5's table gives a single
figure per cell. To say whether a shortcut is worth anything you want an **interval** on the drop.

`credit_base[i]` is pair *i*'s credit (1, 0.5 or 0, from `pair_credit`) with the features as they are, and `credit_cf[i]` the credit of the
**same pair** after a counterfactual. The **drop** of a resample is `mean(credit_base[rows]) - mean(credit_cf[rows])`, the same rows for both:
a pair is a pair, and its two credits belong together. The resampling rows are given: replicate `b` uses `idx[b]`. Return the 2.5th and 97.5th
percentiles of the `B` drops (numpy's default linear interpolation), as a **tuple** `(low, high)` of two Python floats.

<details><summary>💡 Hint 1 — what to think about</summary>

What would be different if the original credits and the counterfactual credits of one replicate were resampled with different rows?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Each row of `idx` is one replicate: it picks pairs, and it picks them once for both credit arrays. Each replicate gives one drop, the
original's mean credit minus the counterfactual's, and the interval is read from the whole list of drops. The two credit arrays with
different shapes, credits that are not one-dimensional, or an `idx` that is not two-dimensional raise `ValueError`.

</details>

In [ ]:
def paired_drop_interval(credit_base, credit_cf, idx):
    """A 95% bootstrap interval for the drop in mean credit caused by a counterfactual.

    Args:
        credit_base, credit_cf: per-pair credits of the same pairs, shape (n,), before and after the counterfactual.
        idx: integer array of shape (B, n); row b is replicate b's resample of the pairs, used for BOTH credit arrays.

    Replicate b's drop is ``mean(credit_base[idx[b]]) - mean(credit_cf[idx[b]])``. Credit arrays of different shapes, credits
    that are not one-dimensional, or an `idx` that is not two-dimensional raise ``ValueError``.

    Returns:
        A tuple ``(low, high)``: the 2.5th and 97.5th percentiles of the B drops (numpy's default linear interpolation), as Python floats.

    Example:
        >>> paired_drop_interval(np.array([1.0, 0.0]), np.array([0.0, 0.0]), np.array([[0, 0], [1, 1], [0, 1]]))
        (0.025, 0.975)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_paired_drop_interval() -> None:
    base = np.array([1.0, 0.5, 0.0, 1.0])
    cf = np.array([0.0, 0.5, 0.0, 0.0])
    idx = np.array([[0, 1, 2, 3], [0, 0, 0, 0], [1, 2, 1, 2], [3, 3, 0, 1], [0, 1, 1, 1]])      # drops 0.5, 1, 0, 0.75, 0.25
    result = paired_drop_interval(base, cf, idx)
    assert type(result) is tuple and len(result) == 2, f"return a tuple (low, high); got a {type(result).__name__}"
    lo, hi = result
    assert abs(lo - 0.025) < 1e-12 and abs(hi - 0.975) < 1e-12, (
        f"got ({lo!r}, {hi!r}). The five replicates' drops are 0.5, 1, 0, 0.75 and 0.25 (original mean minus counterfactual mean, "
        "the same rows for both); take their 2.5th and 97.5th percentiles with linear interpolation.")
    same = np.array([1.0, 0.0, 0.5, 1.0, 0.0, 0.5])
    rows = np.random.default_rng(5).integers(0, 6, (30, 6))
    lo, hi = paired_drop_interval(same, same.copy(), rows)
    assert abs(lo) < 1e-12 and abs(hi) < 1e-12, (
        f"got ({lo!r}, {hi!r}) for a counterfactual that changed nothing. If every pair earns the same credit before and after, the drop of "
        "every replicate is exactly 0: index both arrays with the SAME row, never a different one.")
    flipped = paired_drop_interval(cf, base, idx)
    assert abs(flipped[0] + 0.975) < 1e-12 and abs(flipped[1] + 0.025) < 1e-12, (
        f"got {flipped} with the two credit arrays exchanged; the drop is original minus counterfactual, so it changes sign and the interval flips")
    assert type(lo) is float and type(hi) is float, f"return Python floats (wrap each with float()); got ({type(lo).__name__}, {type(hi).__name__})"
    rng = np.random.default_rng(10)
    many = rng.integers(0, 3, 25) / 2.0
    lo, hi = paired_drop_interval(many, many.copy(), rng.integers(0, 25, (60, 25)))
    assert abs(lo) < 1e-12 and abs(hi) < 1e-12, (
        f"got ({lo!r}, {hi!r}) for a counterfactual that changed no credit, on 60 random replicates: every replicate's drop is exactly 0 when "
        "both arrays are indexed with the same row")
    big_base = rng.integers(0, 3, 120) / 2.0
    big_cf = np.where(rng.random(120) < 0.3, rng.integers(0, 3, 120) / 2.0, big_base)
    big_idx = rng.integers(0, 120, (300, 120))
    drops = np.array([sum(big_base[i] for i in row) / 120 - sum(big_cf[i] for i in row) / 120 for row in big_idx])
    got = paired_drop_interval(big_base, big_cf, big_idx)
    assert all(abs(g - x) < 1e-12 for g, x in zip(got, np.percentile(drops, [2.5, 97.5]))), (
        f"got {got}, a plain loop over the replicates gives {tuple(np.percentile(drops, [2.5, 97.5]))}: replicate b is mean(base[idx[b]]) - mean(cf[idx[b]]), "
        "one row for both")
    small_base, small_cf = rng.integers(0, 3, 40) / 2.0, rng.integers(0, 3, 40) / 2.0
    for b in (41, 400):
        small_idx = rng.integers(0, 40, (b, 40))
        drops = np.array([sum(small_base[i] for i in row) / 40 - sum(small_cf[i] for i in row) / 40 for row in small_idx])
        got, want = paired_drop_interval(small_base, small_cf, small_idx), np.percentile(drops, [2.5, 97.5])
        assert all(abs(g - x) < 1e-12 for g, x in zip(got, want)) and got[0] <= got[1], (
            f"B={b}: got {got}, expected {tuple(want)}: the 2.5th and 97.5th percentiles of the drops with numpy's default linear interpolation, "
            "not 5/95, not nearest-rank, not mean +/- 1.96 standard deviations")
    for bad in ((base, cf[:3], idx), (np.zeros((2, 2)), np.zeros((2, 2)), idx), (np.zeros((2, 2, 2)), np.zeros((2, 2, 2)), idx),
                (np.array(0.5), np.array(0.5), idx), (base, cf, idx[0]), (base, cf, np.zeros((2, 2, 2), dtype=int))):
        assert _raises_value_error(paired_drop_interval, *bad), (
            "credits of different shapes, credits that are not one-dimensional and an idx that is not two-dimensional must raise ValueError")
    print("exercise 5 looks right")


_try("exercise 5", _check_paired_drop_interval)

In [ ]:
def draw_indices(rng, n, b=BOOT):
    """Given: the resampling indices of one interval, b rows of n pairs drawn with replacement."""
    return rng.integers(0, n, (b, n))


def interval_verdict(low, high):
    """Given: where a drop interval sits against zero: all above, all below, or across it."""
    return "above zero (accuracy reliably falls)" if low > 0 else "below zero (accuracy reliably rises)" if high < 0 else "straddles zero (no reliable change)"


def _show_intervals() -> None:
    better, worse = gold_pairs(TEST)
    idx = draw_indices(np.random.default_rng(SEED + 1), N_TEST)
    print(f"drop in pairwise accuracy against the gold reward, with a 95% paired-bootstrap interval ({BOOT} replicates, {N_TEST} pairs):\n")
    verdicts = {}
    for name, w in MODELS.items():
        credit = pair_credit(better @ w, worse @ w)
        for label, column in (("length", LENGTH), ("format", FORMAT)):
            for mode in ("swap", "equalise"):
                nb, nw = counterfactual_pairs(better, worse, column, mode)
                cf = pair_credit(nb @ w, nw @ w)
                lo, hi = paired_drop_interval(credit, cf, idx)
                verdicts[(name, label, mode)] = interval_verdict(lo, hi)
                print(f"  {name:16s}{label + ' ' + mode:18s}drop {credit.mean() - cf.mean():+.3f}   95% interval ({lo:+.3f}, {hi:+.3f})   {verdicts[(name, label, mode)]}")
    print("\nwhere each model's length-swap interval sits against zero:")
    for name in MODELS:
        print(f"  {name:16s}fitted length weight {MODELS[name][LENGTH]:+.2f}; the interval lies {verdicts[(name, 'length', 'swap')]}")
    print("an interval that straddles zero is not a finding of no effect, and one below zero is a finding that accuracy rose: read the sign as well as the width")


_try("intervals", _show_intervals, needs=("exercise 3", "exercise 5"))

### One world, or the typical one?

Everything above is one seed. The bootstrap interval covers the noise of drawing these test pairs for these two fitted models;
it does not cover the noise of drawing the training pairs, which moves the fitted weights. The next cell repeats the whole thing
on fresh worlds: new answers, new annotators' coin flips, two newly fitted models, and the length-swap drop for each, and it says whether
the world above lies inside the range the fresh worlds span. Nothing is picked: the numbers are what the code prints.

In [ ]:
def _replicates() -> None:
    swayed_drops, control_drops = [], []
    for r in range(REPLICATES):
        rng = np.random.default_rng(SEED + 1000 + r)
        train, test = make_pairs(rng, N_TRAIN), make_pairs(rng, N_TEST)
        better, worse = gold_pairs(test)
        nb, nw = counterfactual_pairs(better, worse, LENGTH, "swap")
        for key, drops in ((train.swayed_a_wins, swayed_drops), (train.control_a_wins, control_drops)):
            w, _ = fit_bt(np.subtract(*orient(train.feats_a, train.feats_b, key)))
            drops.append(pairwise_accuracy(better @ w, worse @ w) - pairwise_accuracy(nb @ w, nw @ w))
    swayed_drops, control_drops = np.array(swayed_drops), np.array(control_drops)
    print(f"length-swap drop in accuracy against the gold reward over {REPLICATES} fresh worlds:")
    print(f"  swayed-trained   mean {swayed_drops.mean():+.3f}   range ({swayed_drops.min():+.3f}, {swayed_drops.max():+.3f})")
    print(f"  control-trained  mean {control_drops.mean():+.3f}   range ({control_drops.min():+.3f}, {control_drops.max():+.3f})")
    print(f"the swayed model's drop is larger than the control's in {int(np.sum(swayed_drops > control_drops))} of {REPLICATES} worlds")
    better, worse = gold_pairs(TEST)
    nb, nw = counterfactual_pairs(better, worse, LENGTH, "swap")
    headline = {name: pairwise_accuracy(better @ w, worse @ w) - pairwise_accuracy(nb @ w, nw @ w) for name, w in MODELS.items()}
    spans = {"swayed-trained": swayed_drops, "control-trained": control_drops}
    inside = {name: bool(spans[name].min() <= headline[name] <= spans[name].max()) for name in MODELS}
    for name in MODELS:
        print(f"the world of sections 4 to 7 gave {name} a length-swap drop of {headline[name]:+.3f}; inside the range of the fresh worlds: {inside[name]}")
    if not all(inside.values()):
        print("so at least one single-world figure above is an unusual draw: weigh the fresh-world rows more than that world's own")


_try("replicates", _replicates, needs=("exercise 2", "exercise 3"))

## 8. Self-check

1. At the weights your fit reports as converged, your `bt_loss_grad` returns a gradient that is far from zero.
   Which is the most likely cause?
   - (a) The fit is stuck at a saddle point, which is normal for a logistic loss.
   - (b) The loss is convex, so a nonzero gradient at the minimum is expected.
   - (c) Your loss and gradient do not describe the same function the fit minimised: a missing penalty term, or a sum where the fit uses a mean.
   - (d) The training set was too small for the gradient to vanish.

2. A reward model scores well on held-out pairs labelled by its own kind of annotators. Swapping the length between the better and the worse answer
   of gold-ordered pairs lowers its accuracy a lot, and swapping a content readout lowers it too. What is the sound reading?
   - (a) The model leans on length, and length differs between better and worse answers, so the swap reverses part of its ranking; the content readout's drop is the reference for what a real signal costs.
   - (b) Any drop under a counterfactual proves the model is broken.
   - (c) The counterfactual changed the pairs' gold order, so the accuracy is no longer meaningful.
   - (d) Equalising a feature is the same as deleting it, so the two modes must agree.

3. As n grows in best-of-n, the learned reward of the winners keeps climbing while their gold reward climbs more slowly. What explains the widening gap?
   - (a) Larger pools contain worse candidates on average.
   - (b) The gold reward is a noisier measurement than the learned one.
   - (c) Best-of-n changes the model's weights as it runs.
   - (d) The larger the pool, the more the selection picks candidates whose learned score is high partly through noise and shortcut features that the gold reward ignores.

4. Why does `paired_drop_interval` index the original credits and the counterfactual credits with the same rows?
   - (a) Different rows would make the replicate means differ in size.
   - (b) The drop is a difference within each pair; the two credits of one pair are strongly related, and resampling them separately would add noise from unrelated pairs.
   - (c) numpy cannot index two arrays with different rows.
   - (d) The interval would then be centred on zero.

Answers come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
if __name__ == "__main__":
    print("a fact for question 3, printed here rather than taken on trust (this uses no code of yours):")
    for _name, _w in MODELS.items():
        _score = POOL_FEATS @ _w
        _r_gold = np.corrcoef(_score, POOL_QUALITY.reshape(-1))[0, 1]
        _r_len = np.corrcoef(_score, POOL_FEATS[:, LENGTH])[0, 1]
        print(f"  measured: over the {POOL_PROMPTS * POOL_SIZE} pool candidates the {_name} score correlates {_r_gold:.2f} with the gold reward and {_r_len:.2f} with length")

## 9. Common mistakes

* **Quoting one accuracy.** It depends on whose labels grade it; section 4 prints three for each model, so compare them.
* **A loss and a gradient that disagree.** A model can train on a gradient that is wrong; the stationarity check in exercise 1 is the
  fit saying which is right.
* **A counterfactual that mutates its input.** Section 5 reuses the same `better` and `worse` arrays for every case; a changed original corrupts every later row.
* **Reading a drop as a verdict.** A swap costs accuracy according to the feature's weight times how differently the feature sits in
  better and worse answers (the table after section 5 prints both and their product). A feature with a small or negative weight can leave
  accuracy unchanged or raise it, and a feature unrelated to quality costs nothing to swap however large its weight: read the format rows
  of that table for the weight, the gap and the swap change side by side, and look at the equalise column for the model's use of it.
  Compare with the content readout's row, put an interval on the drop (exercise 5), and read where the interval sits against zero:
  above, across, or below.
* **Reading best-of-n by the learned reward alone.** It climbs by construction (a score that tracks the gold reward at less than 1 gains
  more on itself than on the gold reward); the gold column (or a held-out human check) is the one that says whether the pick was any good.

## 10. What this lesson does NOT cover

* **Real reward models.** The model here is linear in six features. A reward model built on a language model has no weights you can read, which is
  why the counterfactual audit exists; its shortcuts are found by changing real answers (longer, reformatted) and rescoring them.
* **Real annotators.** The swayed annotators are a script. Published work (`claims.yaml`, as of 12 November 2026) reports that length correlates with
  preference labels and reward-model scores; how much, and in which datasets, is for those sources, not this simulation.
* **Generative reward models and LLM judges.** The catalog item names them; T10-L07 audits a judge. This lesson fits and audits the classic scalar model.
* **The benchmarks themselves.** RewardBench 2 and M-RewardBench are named for what they measure (`claims.yaml`); this lesson does not run them,
  needs no network and loads no model. Use the instruments here on any reward model you can score.
* **Over-optimisation laws.** Gao et al. fit functional forms to gold-reward curves under best-of-n and RL; section 6 shows one such curve, it does not fit them.

In [ ]:
def _recap() -> None:
    better, worse = gold_pairs(TEST)
    print("--- what you built, measured ---")
    for name, w in MODELS.items():
        base = pairwise_accuracy(better @ w, worse @ w)
        nb, nw = counterfactual_pairs(better, worse, LENGTH, "swap")
        learned = (POOL_FEATS @ w).reshape(POOL_PROMPTS, POOL_SIZE)
        _, gold_top = best_of_n(learned, POOL_QUALITY, POOL_SIZE)
        print(f"  {name:16s}accuracy vs gold {base:.3f}; after a length swap {pairwise_accuracy(nb @ w, nw @ w):.3f}; "
              f"gold reward of best-of-{POOL_SIZE} winners {gold_top:.2f}")
    print("  every number above was computed by the code you just ran")


if __name__ == "__main__":
    _try("the closing recap", _recap, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "", function) else ""
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero, so a green exit code can never
    paper over it. In a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.